In [0]:
# ============================================================
# 0. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

# ============================================================
# 1. IMPORTS
# ============================================================
import json
import pyspark.sql.functions as f
from common_utils.generic_functions import  casting
from common_utils.logging import get_logger

# ============================================================
# 2. LOGGER
# ============================================================
logger = get_logger("dim_order.ipynb")
logger.info("Session for dim_order started")

# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",config_path
)
with open(config_path,"r") as file:
    config = json.load(file)

logger.info("Gold configuration loaded successfully")



# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================
source_config = config["source"]
target_config= config["target"]

source_name = config["source"]["source_name"]
source_table = config["source"]["source_table"]
target_table = config["target"]["target_table"]

logger.info(
    "[%s] Source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)
# ============================================================
# 5. READ SOURCE TABLE
# ============================================================

logger.info(
    "[%s] Reading order from %s",
    source_name,
    source_table
)

order_df = spark.read.table(source_table)

logger.info(
    "[%s] order table loaded successfully",
    source_name
)


# ============================================================
# 6. TRANSFORM CUSTOMER
# ============================================================


def transform_order(df,config):
    logger.info(
        "[%s] Starting dim_customer transformation",
        source_name
    )
    

    # --------------------------------------------------------
    # Selecting required columns
    # --------------------------------------------------------

    df = df.select("order_number" , "customer_id", "order_timestamp" , "number_of_line_items")


    
    # --------------------------------------------------------
    # converting order timestamp
    # --------------------------------------------------------
    df = df.withColumn("order_timestamp",f.to_timestamp(f.col("order_timestamp")))

    # --------------------------------------------------------
    # Creating order__date_key
    # --------------------------------------------------------

    df = df.withColumn("order_date_key",f.date_format(f.col("order_timestamp"),config["transform"]["order_date_key_format"]).cast("int"))




    # --------------------------------------------------------
    # creating timestamp
    # --------------------------------------------------------

    df = df.withColumn("order_timestamp",f.to_timestamp(f.col("order_timestamp")))
    # --------------------------------------------------------
    # Cast final columns
    # --------------------------------------------------------

    df = casting(df, config["cast_columns"])

    df = df.select(
        *config["select_columns"]
    )
    # --------------------------------------------------------
    # droping duplicate one item per row for the primary key
    # --------------------------------------------------------

    df = df.dropDuplicates(["order_number"])
    
    logger.info(
        "[%s] dim_order transformation completed",
        source_name
    )

    return df
# ============================================================
# 7. CREATE DIM ORDER
# ============================================================

logger.info(
    "[%s] Starting dim_order creation",
    source_name
)




# ============================================================
# 7. CREATE DIM  ORDER
# ============================================================

logger.info(
    "[%s] Starting dim_order creation",
    source_name
)


#creating dim order
dim_order = transform_order(order_df,config)


# ============================================================
# 8. VALIDATE DATA
# ============================================================
logger.info(
    "[%s] Validating dim_order",
    source_name
)

records_count=dim_order.count()
if records_count==0:
    raise ValueError("no record found in Dim_customer")

duplicate_count = (
    dim_order
    .groupBy("order_number")
    .count()
    .filter(f.col("count") > 1)
    .count()
)
if duplicate_count > 0:
    raise ValueError ("Duplicae detected")


# ============================================================
# 9. DISPLAY
# ============================================================
dim_order.display()




# ============================================================
# 10. CREATE GOLD SCHEMA
# ============================================================
logger.info(
    "[%s] Creating Gold schema if required",
    source_name
)

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS 
    retail_data_platform.gold
    
          """)
# ============================================================
# 11. WRITE GOLD TABLE
# ============================================================

logger.info(
    "[%s] Writing dim_order to %s",
    source_name,
    target_table
)

(dim_order.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table ))

logger.info(
    "[%s] Gold table %s written successfully",
    source_name,
    target_table
)

# ============================================================
# 12. FINAL VALIDATION
# ============================================================

final_count = (
    spark.read
    .table(target_table)
    .count()
)

logger.info(
    "[%s] Final dim_order record count: %s",
    source_name,
    final_count
)

logger.info(
    "[%s] dim_order pipeline completed successfully",
    source_name
)

